# FluView Pulse exploratory analysis

This notebook validates the dashboard's trend, season, geography, and age methods. It treats ILINet outpatient respiratory illness and FluSurv-NET laboratory-confirmed hospitalizations as separate surveillance indicators. Current-season observations are preliminary and can be revised.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"

ili = pd.read_parquet(DATA / "ilinet_metrics.parquet")
hosp = pd.read_parquet(DATA / "flusurv_metrics.parquet")

print(f"ILINet: {len(ili):,} rows, {ili['season'].nunique()} seasons")
print(f"FluSurv-NET: {len(hosp):,} rows, {hosp['season'].nunique()} seasons")
print("Latest surveillance dates:", ili.week_ending.max().date(), hosp.week_ending.max().date())

ILINet: 6,281 rows, 11 seasons
FluSurv-NET: 8,584 rows, 8 seasons
Latest surveillance dates: 2026-09-05 2026-09-12


## Seasonal and geographic checks

The live signal compares the latest trailing three-week mean with the preceding three-week mean. It is a signed descriptive change, not an alert. Same-week empirical percentiles use up to ten prior complete seasons and always travel with their historical sample count.

In [2]:
latest_ili_season = ili.loc[ili.week_ending.notna(), "season_start"].max()
latest_region_rows = (
    ili.query("season_start == @latest_ili_season and geography_type == 'hhs'")
    .sort_values("week_ending")
    .groupby("region", as_index=False)
    .tail(1)
    .sort_values("momentum_3wk", ascending=False)
)

assert latest_region_rows["region"].nunique() == 10
assert latest_region_rows["historical_n"].between(0, 10).all()

px.bar(
    latest_region_rows,
    x="momentum_3wk",
    y="region",
    orientation="h",
    title="Latest signed three-week ILINet momentum by HHS region",
    labels={"momentum_3wk": "Percentage-point change", "region": ""},
)

## Robust z-score sensitivity check

A median/MAD score is retained as EDA only. With few comparable seasons, MAD can be zero and standardized thresholds can imply unsupported precision. The dashboard therefore favors absolute momentum and empirical percentile context.

In [3]:
def robust_zscore(group: pd.Series) -> pd.Series:
    median = group.median()
    mad = (group - median).abs().median()
    if mad == 0 or pd.isna(mad):
        return pd.Series(np.nan, index=group.index)
    return 0.6745 * (group - median) / mad


national = ili.query("region == 'National'").copy()
national["robust_z"] = national.groupby("season_week")["value"].transform(robust_zscore)
comparison_columns = [
    "season",
    "season_week",
    "value",
    "historical_percentile",
    "robust_z",
]
comparison = national[comparison_columns].dropna()
print(comparison[["historical_percentile", "robust_z"]].corr())
print("Undefined robust z-score share:", f"{national.robust_z.isna().mean():.1%}")

                       historical_percentile  robust_z
historical_percentile               1.000000  0.552263
robust_z                            0.552263  1.000000
Undefined robust z-score share: 0.0%


## Hospitalization age comparison

FluSurv-NET age-specific rates are the primary age story. They describe laboratory-confirmed hospitalizations in participating counties across 14 states, not a national census. ILINet age fields are retained only for a secondary provider-subset mix view.

In [4]:
latest_hosp_season = hosp["season_start"].max()
age_query = (
    "season_start == @latest_hosp_season and "
    "region == 'FluSurv-NET' and age_group != 'Overall'"
)
age_latest = (
    hosp.query(age_query)
    .sort_values("week_ending")
    .groupby("age_group", as_index=False)
    .tail(1)
    .sort_values("value", ascending=False)
)

assert age_latest["age_group"].nunique() > 3
px.bar(
    age_latest,
    x="age_group",
    y="value",
    title="Latest weekly FluSurv-NET hospitalization rate by age group",
    labels={"age_group": "Age group", "value": "Rate per 100,000"},
)